# Load in raw Ground Truth Dataframe

In [1]:
import pandas as pd
import numpy as np
import os
file_path = '/project/mclark/For/Bayard/Ground_truthing/IGG4/manual_ground_truth-TUBULES.csv'
df = pd.read_csv(file_path)
df

,Unnamed: 0,TAG,CD138,HLAII,DAPI,Claudin1,CD10,MUC1,DATASET
0,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,1,0,IGG4
1,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,0,1,IGG4
2,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,1,0,1,0,0,1,IGG4
3,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,1,1,IGG4
4,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,1,0,1,0,0,1,IGG4
...,...,...,...,...,...,...,...,...,...
481,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,1,0,1,1,0,0,IGG4
482,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,1,1,IGG4
483,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,1,0,IGG4
484,NaN,/project/mclark/For/Bayard/Ground_truthing/IGG...,0,0,1,0,1,0,IGG4


In [5]:
# Labels to channel signal list NOTE: modify if labels are bad for IGG4

lst2 = [('Proximal Tubules', [('DAPI', 1), ('CD10', 1)]),
 ('Proximal Tubules', [('DAPI', 1), ('CD10', 1), ('CD138', 1)]),
 ('Stressed Proximal Tubules', [('DAPI', 1), ('CD10', 1), ('Claudin1',1)]),
 ('Stressed Proximal Tubules', [('DAPI', 1),('CD10',1),('CD138',1),('Claudin1',1)]),
 ('Distal Tubules', [('DAPI', 1),('MUC1',1)]),
 ('Distal Tubules', [('DAPI', 1), ('MUC1', 1),('CD138',1)]),
 ('Stressed Distal Tubules', [('DAPI', 1), ('MUC1', 1),('Claudin1',1)]),
 ('Stressed Distal Tubules', [('DAPI', 1), ('MUC1', 1), ('CD138', 1),('Claudin1',1)]),
 ('Other Tubules', [('DAPI', 1)]),
 ('Other Tubules', [('DAPI', 1), ('CD138', 1)]),
 ('Other Tubules', [('DAPI',1), ('CD10', 1),('MUC1',1)]),
 ('Other Tubules', [('DAPI',1), ('CD10',1), ('MUC1',1), ('CD138',1)]),
 ('Other Stressed Tubules', [('DAPI', 1), ('Claudin1', 1)]),
 ('Other Stressed Tubules', [('DAPI', 1), ('CD138', 1), ('Claudin1', 1)]),
 ('Other Stressed Tubules', [('DAPI',1), ('CD10',1), ('MUC1',1), ('Claudin1',1)]),
 ('Other Stressed Tubules', [('DAPI',1),('CD10',1), ('MUC1',1), ('CD138',1), ('Claudin1',1)]),
 ('RBC', [('DAPI', 0), ('CD10', 0),('CD138',0), ('HLAII',0),('MUC1',0), ('Claudin1',0)])]

In [6]:
def classify_rows(df, classification_list):
    binary_cols = ['CD10', 'CD138', 'Claudin1', 'HLAII', 'MUC1', 'DAPI']
    
    label_names = []
    label_numbers = []
    
    for _, row in df.iterrows():
        match_found = False
        row_active = set(col for col in binary_cols if row[col] == 1)
        
        for class_idx, (class_name, criteria) in enumerate(classification_list, start=1):
            criteria_cols = {col for col, val in criteria if val == 1}
            
            if row_active == criteria_cols:
                label_names.append(class_name)
                label_numbers.append(class_idx)
                match_found = True
                break
        
        if not match_found:
            label_names.append("unclassified")
            label_numbers.append(-1)
    
    df["label_name"] = label_names
    df["label_number"] = label_numbers
    return df

result_df = classify_rows(df, lst2)

# Count Markers without Matches

In [7]:
def analyze_marker_combinations(df):
    negative_labels = df[df['label_name'] == 'unclassified']

    # Marker columns of interest
    cols = ['CD10', 'Claudin1', 'CD138', 'HLAII', 'MUC1', 'DAPI']

    # Filter to marker columns
    marker_cols = [col for col in df.columns if col in cols]

    # Create compressed combinations of positive markers
    combinations = negative_labels[marker_cols].apply(
        lambda row: '/'.join(col for col, val in row.items() if val > 0),
        axis=1
    )

    # Get value counts of compressed combinations
    combination_counts = combinations.value_counts()

    return combination_counts

results = analyze_marker_combinations(result_df)
print("Combinations and counts for Mistmatched Labels")
print(results)

Combinations and counts for Mistmatched Labels
CD138/CD10/MUC1     2
HLAII/DAPI/CD10     1
CD138/HLAII/DAPI    1
CD138/Claudin1      1
Name: count, dtype: int64


# Classification Counts for Ground Truth

In [8]:
classification_counts = result_df['label_name'].value_counts()
classification_percentages = (classification_counts / classification_counts.sum()) * 100
print("Classification Counts:")
print(classification_counts)
print("\nClassification Percentages:")
print(classification_percentages)

# unclassified = 86

Classification Counts:
label_name
Distal Tubules               192
Proximal Tubules             116
Other Tubules                 81
Other Stressed Tubules        66
Stressed Proximal Tubules     23
unclassified                   5
Stressed Distal Tubules        2
RBC                            1
Name: count, dtype: int64

Classification Percentages:
label_name
Distal Tubules               39.506173
Proximal Tubules             23.868313
Other Tubules                16.666667
Other Stressed Tubules       13.580247
Stressed Proximal Tubules     4.732510
unclassified                  1.028807
Stressed Distal Tubules       0.411523
RBC                           0.205761
Name: count, dtype: float64


In [9]:
# Export Dataframe with Explicit Labels
export_df = result_df[['TAG','DATASET','label_name']]
export_df.to_csv('/project/mclark/For/Bayard/Ground_truthing/IGG4/labels_manual_ground_truth-TUBULES.csv')
export_df

,TAG,DATASET,label_name
0,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Proximal Tubules
1,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Distal Tubules
2,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Distal Tubules
3,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Other Tubules
4,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Distal Tubules
...,...,...,...
481,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Other Stressed Tubules
482,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Other Tubules
483,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Proximal Tubules
484,/project/mclark/For/Bayard/Ground_truthing/IGG...,IGG4,Proximal Tubules
